# Importing Packages

In [7]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
from function import *

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import KNeighborsRegressor
from scipy.stats import uniform, randint
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import StackingRegressor, RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures


from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score


import warnings
warnings.filterwarnings("ignore")

RSEED=42

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Importing the Data

In [8]:
# Import diabetes data
df = pd.read_csv('./../Data/Train.csv')
df.head()

,Place_ID X Date,Date,Place_ID,target,target_min,target_max,target_variance,target_count,precipitable_water_entire_atmosphere,relative_humidity_2m_above_ground,...,L3_SO2_sensor_zenith_angle,L3_SO2_solar_azimuth_angle,L3_SO2_solar_zenith_angle,L3_CH4_CH4_column_volume_mixing_ratio_dry_air,L3_CH4_aerosol_height,L3_CH4_aerosol_optical_depth,L3_CH4_sensor_azimuth_angle,L3_CH4_sensor_zenith_angle,L3_CH4_solar_azimuth_angle,L3_CH4_solar_zenith_angle
0,010Q650 X 2020-01-02,2020-01-02,010Q650,38.0,23.0,53.0,769.50,92,11.000000,60.200001,...,38.593017,-61.752587,22.363665,1793.793579,3227.855469,0.010579,74.481049,37.501499,-62.142639,22.545118
1,010Q650 X 2020-01-03,2020-01-03,010Q650,39.0,25.0,63.0,1319.85,91,14.600000,48.799999,...,59.624912,-67.693509,28.614804,1789.960449,3384.226562,0.015104,75.630043,55.657486,-53.868134,19.293652
2,010Q650 X 2020-01-04,2020-01-04,010Q650,24.0,8.0,56.0,1181.96,96,16.400000,33.400002,...,49.839714,-78.342701,34.296977,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,010Q650 X 2020-01-05,2020-01-05,010Q650,49.0,10.0,55.0,1113.67,96,6.911948,21.300001,...,29.181258,-73.896588,30.545446,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,010Q650 X 2020-01-06,2020-01-06,010Q650,21.0,9.0,52.0,1164.82,95,13.900001,44.700001,...,0.797294,-68.612480,26.899694,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30557 entries, 0 to 30556
Data columns (total 82 columns):
 #   Column                                               Non-Null Count  Dtype  
---  ------                                               --------------  -----  
 0   Place_ID X Date                                      30557 non-null  object 
 1   Date                                                 30557 non-null  object 
 2   Place_ID                                             30557 non-null  object 
 3   target                                               30557 non-null  float64
 4   target_min                                           30557 non-null  float64
 5   target_max                                           30557 non-null  float64
 6   target_variance                                      30557 non-null  float64
 7   target_count                                         30557 non-null  int64  
 8   precipitable_water_entire_atmosphere                 30557 non-nul

In [10]:
df.isnull().sum()

Place_ID X Date                     0
Date                                0
Place_ID                            0
target                              0
target_min                          0
                                ...  
L3_CH4_aerosol_optical_depth    24765
L3_CH4_sensor_azimuth_angle     24765
L3_CH4_sensor_zenith_angle      24765
L3_CH4_solar_azimuth_angle      24765
L3_CH4_solar_zenith_angle       24765
Length: 82, dtype: int64

# EDA 1

In [11]:
# --- 1. GLOBAL CLEANING ---
# Treat 0 as missing data for satellite bands
df = df.replace(0, np.nan)

# --- 2. THE SPLIT (The "Wall") ---
# We split first so we know exactly what data we are working with
y = df["target"] 
X = df.drop("target", axis=1)

# Standard Zindi Split: Train, Validation, and a Final Test set
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42)


# --- 3. DEFINE THE PIPELINE (The "Blueprint") ---
cols_to_drop = ["Place_ID X ", "target_", "ch4"]

# ----- 4. XG Boost settings 

xgb_random_grid = {
    'n_estimators': randint(10, 100), 
    'learning_rate': uniform(0.01, 0.07),
    'subsample': uniform(0.7, 0.2),       # 0.7 to 0.9
    'colsample_bytree': uniform(0.7, 0.2),
    'gamma': [0, 1, 5],
    'reg_lambda': [1, 10]
}

xgb_base = XGBRegressor(random_state=RSEED, n_jobs=-1, tree_method='hist')

xgb_random = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_random_grid,
    n_iter=15,
    cv=3,
    verbose=1,
    random_state=RSEED,
    scoring='neg_root_mean_squared_error'
)

# ----- 5. Stacking settings  

knn_pipe = Pipeline([
    ('knn', KNeighborsRegressor())
])

poly_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=3)),
    ('lin_reg', LinearRegression())
])

rf_model = RandomForestRegressor(n_estimators=100, random_state=RSEED, n_jobs=-1)

estimators = [
    ('rf', rf_model),
    ('knn', knn_pipe)
]

stacking_model = StackingRegressor(
    estimators=estimators,
    final_estimator=poly_pipe, 
    cv=5 
)

# ----- 6. Create the PREPROCESSING PART of the pipeline (Everything except the model)
preprocessing_pipeline = Pipeline([
    ('drop_keywords', FunctionTransformer(lambda x: drop_features(x.copy(), cols_to_drop))),
    ('imputer', GroupedMedianImputer()), # Using the class from your function.py
    ('angles', FunctionTransformer(lambda x: relative_mean_angles(x.copy()))),
    ('amf', FunctionTransformer(lambda x: calculate_air_mass_factors(x.copy()))),
    ('indices', FunctionTransformer(lambda x: calculate_atmospheric_indices(x.copy()))),
    ('cloud_red', FunctionTransformer(lambda x: cloud_fraction_reduction(x.copy()))),
    ('alt_red', FunctionTransformer(lambda x: sensor_altitude_reduction(x.copy()))),
    ('drop_meta', FunctionTransformer(lambda x: x.drop(columns=['Date', 'Place_ID'], errors='ignore'))),
    ('scalar', RobustScaler())
])

# ----- 7. Define your models
models = {
    "KNN": KNeighborsRegressor(n_neighbors=5),
    "XGBoost": xgb_random,
    "Stacking": stacking_model
}

# ----- 8. Loop through and get results for each

for name, model in models.items():
    print(f"\n--- Running {name} ---")
    
    # Create a full pipeline for THIS specific model
    full_pipeline = Pipeline([
        ('preprocessor', preprocessing_pipeline),
        ('regressor', model)
    ])
    
    # Enable pandas output for your custom functions
    full_pipeline.set_output(transform="pandas")
    
    # Fit and Predict
    full_pipeline.fit(X_train, y_train)
    y_pred_train = full_pipeline.predict(X_train)
    y_pred_val = full_pipeline.predict(X_val)
    y_pred_test = full_pipeline.predict(X_test)
    
    # Store and Print Results
    rmse_train = round(root_mean_squared_error(y_train, y_pred_train), 3)
    mae_train = round(mean_absolute_error(y_train, y_pred_train), 3)
    r2_train = round(r2_score(y_train, y_pred_train), 3)

    rmse_val = round(root_mean_squared_error(y_val, y_pred_val), 3)
    mae_val = round(mean_absolute_error(y_val, y_pred_val), 3)
    r2_val = round(r2_score(y_val, y_pred_val), 3)
    
    rmse_test = round(root_mean_squared_error(y_test, y_pred_test), 3)
    mae_test = round(mean_absolute_error(y_test, y_pred_test), 3)
    r2_test = round(r2_score(y_test, y_pred_test), 3)

    print(f"------------------------------------------------")
    print(f"{name} Train RMSE: {rmse_train}")
    print(f"{name} Train MAE: {mae_train}")
    print(f"{name} Train R2: {r2_train}")

    print(f"------------------------------------------------")
    print(f"{name} Validation RMSE: {rmse_val}")
    print(f"{name} Validation MAE: {mae_val}")
    print(f"{name} Validation R2: {r2_val}")

    print(f"------------------------------------------------")
    print(f"{name} Test RMSE: {rmse_test}")
    print(f"{name} Test MAE: {mae_test}")
    print(f"{name} Test R2: {r2_test}")


--- Running KNN ---
------------------------------------------------
KNN Train RMSE: 27.518
KNN Train MAE: 18.391
KNN Train R2: 0.659
------------------------------------------------
KNN Validation RMSE: 32.792
KNN Validation MAE: 22.68
KNN Validation R2: 0.492
------------------------------------------------
KNN Test RMSE: 33.7
KNN Test MAE: 22.554
KNN Test R2: 0.484

--- Running XGBoost ---
Fitting 3 folds for each of 15 candidates, totalling 45 fits
------------------------------------------------
XGBoost Train RMSE: 26.536
XGBoost Train MAE: 17.967
XGBoost Train R2: 0.683
------------------------------------------------
XGBoost Validation RMSE: 29.252
XGBoost Validation MAE: 20.48
XGBoost Validation R2: 0.596
------------------------------------------------
XGBoost Test RMSE: 30.493
XGBoost Test MAE: 20.484
XGBoost Test R2: 0.577

--- Running Stacking ---
------------------------------------------------
Stacking Train RMSE: 23.677
Stacking Train MAE: 8.114
Stacking Train R2: 0.748

In [12]:
# --- 1. GLOBAL CLEANING ---
# Treat 0 as missing data for satellite bands
df = df.replace(0, np.nan)

# --- 2. THE SPLIT (The "Wall") ---
# We split first so we know exactly what data we are working with
y = df["target"] 
X = df.drop("target", axis=1)

# Standard Zindi Split: Train, Validation, and a Final Test set
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42)


# --- 3. DEFINE THE PIPELINE (The "Blueprint") ---
cols_to_drop = ["Place_ID X ", "target_", "ch4"]


model_pipeline = Pipeline([
    # A. Drop unwanted features first
    ('drop_keywords', FunctionTransformer(lambda x: drop_features(x, cols_to_drop))),
    
    # B. Impute while Place_ID is still available
    ('imputer', GroupedMedianImputer()),
    
    # C. Feature Engineering (Your Custom Functions)
    ('angles', FunctionTransformer(relative_mean_angles)),
    ('amf', FunctionTransformer(calculate_air_mass_factors)),
    ('indices', FunctionTransformer(calculate_atmospheric_indices)),
    ('cloud_red', FunctionTransformer(cloud_fraction_reduction)),
    ('alt_red', FunctionTransformer(sensor_altitude_reduction)),
    
    # D. Drop Metadata (Date/ID) so the model only sees numbers
    ('drop_meta', FunctionTransformer(lambda x: x.drop(columns=['Date', 'Place_ID'], errors='ignore'))),
    
    # E. The Model (No Scaling as requested)
    ('KNN', KNeighborsRegressor())
])

# --- 5. FIT AND EVALUATE ---
# This fits the imputer and the model on X_train ONLY
model_pipeline.fit(X_train, y_train)

# Predict
y_pred_train = model_pipeline.predict(X_train)
y_pred_val = model_pipeline.predict(X_val)
y_pred_test = model_pipeline.predict(X_test)

# Results
print(f"--- Final Baseline Results ---")
print(f"------------------------------------------------")
print(f"The RMSE for train is: ", round(root_mean_squared_error(y_train, y_pred_train), 3))
print(f"The MAE for train is: ", round(mean_absolute_error(y_train, y_pred_train), 3))
print(f"The R2 for train is: ", round(r2_score(y_train, y_pred_train), 3))
print(f"------------------------------------------------")
print(f"The RMSE for validation is: ", round(root_mean_squared_error(y_val, y_pred_val), 3))
print(f"The MAE for validation is: ", round(mean_absolute_error(y_val, y_pred_val), 3))
print(f"The R2 for validation is: ", round(r2_score(y_val, y_pred_val), 3))
print(f"------------------------------------------------")
print(f"The RMSE for test is: ", round(root_mean_squared_error(y_test, y_pred_test), 3))
print(f"The MAE for test is: ", round(mean_absolute_error(y_test, y_pred_test), 3))
print(f"The R2 for test is: ", round(r2_score(y_test, y_pred_test), 3))

--- Final Baseline Results ---
------------------------------------------------
The RMSE for train is:  36.246
The MAE for train is:  25.619
The R2 for train is:  0.408
------------------------------------------------
The RMSE for validation is:  43.433
The MAE for validation is:  31.236
The R2 for validation is:  0.109
------------------------------------------------
The RMSE for test is:  44.076
The MAE for test is:  31.121
The R2 for test is:  0.117


# Extra Code

In [13]:
xgb_random_grid = {
    'n_estimators': randint(10, 100), 
    'learning_rate': uniform(0.01, 0.07),
    'subsample': uniform(0.7, 0.2),       # 0.7 to 0.9
    'colsample_bytree': uniform(0.7, 0.2),
    'gamma': [0, 1, 5],
    'reg_lambda': [1, 10]
}

xgb_base = XGBRegressor(random_state=RSEED, n_jobs=-1, tree_method='hist')

xgb_random = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_random_grid,
    n_iter=15,
    cv=3,
    verbose=1,
    random_state=RSEED,
    scoring='neg_root_mean_squared_error'
)


In [14]:
# 1. Filter: Keep only Place_IDs that have at least 23 entries
# We group by the column (axis=0 is default), not the headers (axis=1)
df = df.replace(0, np.nan)
df = df.groupby('Place_ID', as_index=False).filter(lambda x: len(x) >= 23)

# 2. Sort: Ensure time-series order for interpolation to make sense
# If 'Date' isn't a datetime object yet, you might want: df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Place_ID', 'Date'])

# 3. Interpolate: Fill missing values within each group
# We use group_keys=False to prevent the Place_ID from becoming a redundant index
df = df.groupby('Place_ID', as_index=False).apply(lambda x: x.interpolate(limit_direction='both'))

# Check the results
df['Place_ID'].value_counts()

# --- 2. THE SPLIT (The "Wall") ---
# We split first so we know exactly what data we are working with
y = df["target"] 
X = df.drop("target", axis=1)

# Standard Zindi Split: Train, Validation, and a Final Test set
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42)

# --- 4. DEFINE THE PIPELINE (The "Blueprint") ---
cols_to_drop = ["Place_ID X ", "target_", "ch4"]

model_pipeline = Pipeline([
    # A. Drop unwanted features first
    ('drop_keywords', FunctionTransformer(lambda x: drop_features(x, cols_to_drop))),
    
    # C. Feature Engineering (Your Custom Functions)
    ('angles', FunctionTransformer(relative_mean_angles)),
    ('amf', FunctionTransformer(calculate_air_mass_factors)),
    ('indices', FunctionTransformer(calculate_atmospheric_indices)),
    ('cloud_red', FunctionTransformer(cloud_fraction_reduction)),
    ('alt_red', FunctionTransformer(sensor_altitude_reduction)),
    
    # D. Drop Metadata (Date/ID) so the model only sees numbers
    ('drop_meta', FunctionTransformer(lambda x: x.drop(columns=['Date', 'Place_ID'], errors='ignore'))),
    
    # E. The Model (No Scaling as requested)
    ('lin_reg', LinearRegression())
])

# --- 5. FIT AND EVALUATE ---
# This fits the imputer and the model on X_train ONLY
model_pipeline.fit(X_train, y_train)

# Predict
y_pred_train = model_pipeline.predict(X_train)
y_pred_val = model_pipeline.predict(X_val)

# Results
print(f"--- Final Baseline Results ---")
print(f"The RMSE for train is: ", round(root_mean_squared_error(y_train, y_pred_train), 3))
print(f"The RMSE for validation is: ", round(root_mean_squared_error(y_val, y_pred_val), 3))
print(f"The MAE for validation is: ", round(mean_absolute_error(y_val, y_pred_val), 3))
print(f"The R2 for validation is: ", round(r2_score(y_val, y_pred_val), 3))

--- Final Baseline Results ---
The RMSE for train is:  36.326
The RMSE for validation is:  418.513
The MAE for validation is:  31.837
The R2 for validation is:  -79.867


In [15]:
# --- 1. GLOBAL CLEANING ---
df = df.replace(0, np.nan)

# --- NEW STEP: FILTER FOR DATA DENSITY ---
# Count how many rows each Place_ID has
counts = df['Place_ID'].value_counts()
# Identify locations with at least 40 data points
dense_places = counts[counts >= 23].index

# Option: Filter the entire DF or just the Training set (Filtering entire DF here for stability)
df = df[df['Place_ID'].isin(dense_places)]

# --- 2. THE SPLIT ---
y = df["target"] 
X = df.drop("target", axis=1)

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42)

# --- 3. THE CUSTOM IMPUTER (With set_output fix) ---
class GroupedMedianImputer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.group_medians = None
        self.global_medians = None

    def fit(self, X, y=None):
        self.group_medians = X.groupby('Place_ID').median(numeric_only=True)
        self.global_medians = X.median(numeric_only=True)
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.group_medians.columns:
            X[col] = X[col].fillna(X['Place_ID'].map(self.group_medians[col]))
            X[col] = X[col].fillna(self.global_medians[col])
        # FINAL SAFETY: Linear Regression explodes on NaNs or Infs
        X = X.fillna(0).replace([np.inf, -np.inf], 0)
        return X
    
    def set_output(self, transform=None):
        return self

# --- 4. DEFINE THE PIPELINE ---
cols_to_drop = ["Place_ID X ", "target_", "ch4"]

model_pipeline = Pipeline([
    ('drop_keywords', FunctionTransformer(lambda x: drop_features(x.copy(), cols_to_drop))),
    ('imputer', GroupedMedianImputer()),
    ('angles', FunctionTransformer(lambda x: relative_mean_angles(x.copy()))),
    ('amf', FunctionTransformer(lambda x: calculate_air_mass_factors(x.copy()))),
    ('indices', FunctionTransformer(lambda x: calculate_atmospheric_indices(x.copy()))),
    ('cloud_red', FunctionTransformer(lambda x: cloud_fraction_reduction(x.copy()))),
    ('alt_red', FunctionTransformer(lambda x: sensor_altitude_reduction(x.copy()))),
    ('drop_meta', FunctionTransformer(lambda x: x.drop(columns=['Date', 'Place_ID'], errors='ignore'))),
    ('lin_reg', LinearRegression())
])

# Force pipeline to handle DataFrames
model_pipeline.set_output(transform="pandas")

# --- 5. FIT AND EVALUATE ---
model_pipeline.fit(X_train, y_train)

y_pred_train = model_pipeline.predict(X_train)
y_pred_val = model_pipeline.predict(X_val)

print(f"--- Results with 40+ Density Filter ---")
print(f"Train RMSE: {round(root_mean_squared_error(y_train, y_pred_train), 3)}")
print(f"Val RMSE: {round(root_mean_squared_error(y_val, y_pred_val), 3)}")
print(f"R2 Score: {round(r2_score(y_val, y_pred_val), 3)}")

--- Results with 40+ Density Filter ---
Train RMSE: 36.326
Val RMSE: 418.513
R2 Score: -79.867


In [16]:
# Import diabetes data
df = pd.read_csv('./../Data/Test.csv')
df.head()
df['Place_ID'].value_counts()

Place_ID
0OS9LVX    94
NJNT1L4    94
MNOX0WS    94
N04QCJO    94
N0O5ZLI    94
           ..
8CPKUI4    35
HYKM9JF    29
DES3SOP    26
RG3VKJB    25
M2NF5RJ    24
Name: count, Length: 179, dtype: int64